# GUS02B: Complete Database Pipeline

Combined notebook that builds the full GeoTERYT database from scratch:

1. Load base database (from GUS01E) with geometries
2. Set up record hierarchy (parent/child links, fake old-voivodeship records)
3. Load all BDL + Census subject data
4. Load historical cross-table pickles (from GUS02A) as DataSeries
5. Resolve historical TERYT codes
6. Extend year range to 1986
7. Create ALL merged subjects (auto-merge + custom M_ subjects)
8. Build cross tables for all subjects
9. Extract population, classify urban/rural, code dimension labels
10. Save complete database

**Input**: `geoteryt_complete_geom_OW.pkl` (from GUS01E)

**Output**: `geoteryt_complete_final.pkl`

In [ ]:
# =============================================================================
# STEP 1: Imports and Path Setup
# =============================================================================
import os
import sys
from pathlib import Path
import importlib
import gc
import pickle

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f'Repository root: {repo_root}')
print(f'Data root: {data_root}')
print(f'GUS root: {gus_root}')

In [ ]:
# =============================================================================
# STEP 2: Load Base Database (from GUS01E)
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

In [ ]:
# =============================================================================
# STEP 2A.1: Fix codes for Warsaw
# =============================================================================

print(f"Old codes: {db.get_by_teryt_id("1465011").code_by_year}")
for year in range(1999, 2002):
    db.get_by_teryt_id("1465011").code_by_year[year] = '1431001'
for year in range(2002,2025):
    db.get_by_teryt_id("1431001").code_by_year[year] = '1465011'
    db.get_by_teryt_id('1412031').code_by_year[year] = '1465158'
print(f"Corrected codes: {db.get_by_teryt_id("1465011").code_by_year}")
print(f"Corrected codes: {db.get_by_teryt_id("1431001").code_by_year}")

db.get_by_teryt_id("1465011").historical_codes = ['1465011', '1431001']
db.get_by_teryt_id("1431001").historical_codes = ['1465011', '1431001']

In [ ]:
# Add wesola record

rec = db.get_by_teryt_id('1465148')
wesola_rec = gtdb.TERYTRecord(teryt_id= '1465158',
                              name = 'Wesoła',
                              name_dod = rec.name_dod,
                              level = rec.level,
                              kind = rec.kind
                              )
wesola_rec.code_by_year = db.get_by_teryt_id("1412031").code_by_year
wesola_rec.historical_codes = ['1412031', '1465158', '1431201']
wesola_rec.geometry = db.get_by_teryt_id("1412031").geometry
db._records[wesola_rec.teryt_id] = wesola_rec

for year in range(2002, 2025):
    db._by_year[year].add(wesola_rec.teryt_id)

db._by_name['wesoła'].add(wesola_rec.teryt_id)
db._by_level[wesola_rec.level].add(wesola_rec.teryt_id)
db._by_kind[wesola_rec.kind].add(wesola_rec.teryt_id)
db._by_voivodeship[wesola_rec.woj].add(wesola_rec.teryt_id)


In [ ]:
# =============================================================================
# STEP 2A.2: Fix codes for Warsaw
# =============================================================================
warsaw_districts = ['1431011','1431021','1431031','1431041','1431121','1431131','1431141','1431151','1431161','1431171','1431181']
for district_id in warsaw_districts:
    rec = db.get_by_teryt_id(district_id)
    new_teryt = rec.teryt_id[:-1] + '8'
    new_rec = gtdb.TERYTRecord(
        teryt_id=new_teryt,
        name=rec.name,
        name_dod = rec.name_dod,
        level=rec.level,
        kind=rec.kind)
    new_rec.code_by_year = rec.code_by_year.copy()
    new_rec.historical_codes = rec.historical_codes.copy()
    new_rec.historical_codes.append(new_teryt)
    new_rec.geometry = rec.geometry
    
    db._records[new_rec.teryt_id] = new_rec
    db._records.pop(rec.teryt_id)
    for year in range(1999,2002):
        db._by_year[year].remove(rec.teryt_id)
        db._by_year[year].add(new_rec.teryt_id)
    for name in db._by_name:
        if rec.teryt_id in db._by_name[name]:
            db._by_name[name].remove(rec.teryt_id)
            db._by_name[name].add(new_rec.teryt_id)
    for level in db._by_level:
        if rec.teryt_id in db._by_level[level]:
            db._by_level[level].remove(rec.teryt_id)
            db._by_level[level].add(new_rec.teryt_id)
    for kind in db._by_kind:
        if rec.teryt_id in db._by_kind[kind]:
            db._by_kind[kind].remove(rec.teryt_id)
            db._by_kind[kind].add(new_rec.teryt_id)
    for voi in db._by_voivodeship:
        if rec.teryt_id in db._by_voivodeship[voi]:
            db._by_voivodeship[voi].remove(rec.teryt_id)
            db._by_voivodeship[voi].add(new_rec.teryt_id)

In [ ]:
# =============================================================================
# STEP 3: Link Children to Parents
# =============================================================================
db.link_children_to_parents()

In [ ]:
# =============================================================================
# STEP 4: Load Pre-1999 Voivodeship Boundaries + Create Fake TERYTRecords
#         + Mazowieckie NUTS Split (1300000 / 1500000)
# =============================================================================
from shapely.ops import unary_union

pre1999_path = geo_root / 'voivodeships_1975_49_clean_EPSG2180.geojson'
if not pre1999_path.exists():
    raise FileNotFoundError(f'Pre-1999 boundaries not found: {pre1999_path}')

pre1999_gdf = gpd.read_file(pre1999_path)
pre1999_gdf = pre1999_gdf.to_crs('EPSG:2180')
pre1999_gdf['teryt_id'] = pre1999_gdf['voiv_id'].apply(lambda x: str(100-x) + '00000')

# Store old voivodeships GeoDataFrame on the database
db.set_old_voivodship_gdf(pre1999_gdf)

# All snapshot years (to populate year-keyed dicts)
ALL_YEARS = sorted(db._by_year.keys())  # typically 1999-2025

# --- OLD VOIVODESHIPS ---
# Children are at gmina level (level 6) based on the 1999 snapshot,
# constant for all years.
old_voiv_ids = []
for idx, row in pre1999_gdf.iterrows():
    record = gtdb.TERYTRecord(teryt_id=row['teryt_id'],
                              name=str(row['name']).capitalize())
    record.level = 2
    record.kind = 0
    record.geometry = row.geometry
    # Parent = country for all years
    record.parent_id = {y: '0000000' for y in ALL_YEARS}
    # Children = all gminas (level 6) that belong to this old voivodeship
    # (constant for all years, based on old_woj matching)
    children = []
    for tid, rec in db._records.items():
        if rec.old_woj and str(rec.old_woj).lower() == str(row['name']).lower():
            children.append(tid)
    record.children_ids = {y: children for y in ALL_YEARS}
    db._records[record.teryt_id] = record
    old_voiv_ids.append(record.teryt_id)

print(f'Added {len(pre1999_gdf)} old voivodeship records to database')

# --- REGION WARSZAWSKI STOLECZNY ---
record_WAW = gtdb.TERYTRecord(teryt_id='1300000', name='Warszawski stołeczny')
record_WAW.level = 2
record_WAW.kind = 0
record_WAW.parent_id = {y: '0000000' for y in ALL_YEARS}
waw_powiat_children = ['1421000','1418000','1432000','1434000','1417000',
                       '1414000','1405000','1408000','1412000','1431000']
record_WAW.children_ids = {y: waw_powiat_children for y in ALL_YEARS}
waw_geoms = [db._records[tid].geometry for tid in waw_powiat_children
             if tid in db._records and db._records[tid].geometry is not None]
if waw_geoms:
    record_WAW.geometry = unary_union(waw_geoms)
db._records['1300000'] = record_WAW

# --- REGION MAZOWIECKI REGIONALNY ---
record_MAZ = gtdb.TERYTRecord(teryt_id='1500000', name='Mazowiecki regionalny')
record_MAZ.level = 2
record_MAZ.kind = 0
record_MAZ.parent_id = {y: '0000000' for y in ALL_YEARS}
mazowieckie_record = db._records.get('1200000')
if mazowieckie_record:
    # Use 1999 snapshot of Mazowieckie children as baseline
    all_maz_children = set(mazowieckie_record.get_children(1999))
    waw_children = set(waw_powiat_children)
    maz_children = sorted(all_maz_children - waw_children)
    record_MAZ.children_ids = {y: maz_children for y in ALL_YEARS}
    maz_geoms = [db._records[tid].geometry for tid in maz_children
                 if tid in db._records and db._records[tid].geometry is not None]
    if maz_geoms:
        record_MAZ.geometry = unary_union(maz_geoms)
db._records['1500000'] = record_MAZ

# --- Set 'old' and 'nuts' keys on country record ---
root = db._records.get('0000000')
if root:
    root.children_ids['old'] = sorted(old_voiv_ids)
    # NUTS: new voivodeships with Maz split (1300000+1500000 instead of 1400000)
    new_voiv_ids = sorted(f'{c:02d}00000' for c in range(2, 33, 2))
    nuts_voiv_ids = sorted(
        [v for v in new_voiv_ids if v != '1400000'] + ['1300000', '1500000']
    )
    root.children_ids['nuts'] = nuts_voiv_ids

print(f'Added REGION WARSZAWSKI STOLECZNY (1300000): {len(record_WAW.get_children(1999))} children')
print(f'Added REGION MAZOWIECKI REGIONALNY (1500000): {len(record_MAZ.get_children(1999))} children')
print(f'Country "old" children: {len(root.children_ids.get("old", []))} old voivodeships')
print(f'Country "nuts" children: {len(root.children_ids.get("nuts", []))} NUTS voivodeships')
print(f'Total records in database: {len(db._records)}')

In [ ]:
# =============================================================================
# STEP 5: Load BDL + Census Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / 'data' / 'bdl_demographic_data.csv', encoding='utf-8')
df_educ_add_1 = pd.read_csv(gus_root / 'data' / 'P2350_BDL_educ_1995_2020 [remove data for 2000].csv',
                             encoding='utf-8', sep=';')
df_educ_add_2 = pd.read_csv(gus_root / 'data' / 'P4092_BDL_educ_2010_2024.csv',
                             encoding='utf-8', sep=';')
df_c_1988 = pd.read_csv(gus_root / 'data' / 'census_data' / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / 'data' / 'census_data' / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / 'data' / 'census_data' / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / 'data' / 'census_data' / 'NSP2021_data.csv', encoding='utf-8')
df_c_2021_add = pd.read_csv(gus_root / 'data' / 'census_data' / 'P4315_Census_2021_educ_sex.csv',
                             encoding='utf-8', sep=';')

df_variables = pd.read_csv(gus_root / 'metadata' / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / 'metadata' / 'census_meta.csv', encoding='utf-8')

# Fix census 1988 year typo (BDL API returns 1998 instead of 1988)
# NOTE: 'year' is the FIRST key in the dict, so no leading comma
for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values'] = df_c_1988.at[id, 'values'].replace("'year': '1998'", "'year': '1988'")
df_c_variables['years'] = df_c_variables['years'].str.replace('[1998]', '[1988]', regex=False)
df_c_variables['n4'] = None
df_c_variables['n5'] = None

print(f'df_demographic: {df_demographic.shape}')
print(f'df_c_1988: {df_c_1988.shape}')
print(f'df_c_2002: {df_c_2002.shape}')
print(f'df_c_2011: {df_c_2011.shape}')
print(f'df_c_2021: {df_c_2021.shape}')
print(f'df_c_2021_add (P4315): {df_c_2021_add.shape}')
print(f'df_educ_add_1 (P2350): {df_educ_add_1.shape}')
print(f'df_educ_add_2 (P4092): {df_educ_add_2.shape}')

In [ ]:
# =============================================================================
# STEP 6: Collect Subject IDs and Names
# =============================================================================
IGNORED_SUBJECTS = {'P4345', 'P3310', 'P1336', 'P2914'}

subject_ids = {'BDL': [], 'Census': {'1988': [], '2002': [], '2011': [], '2021': []}}
subject_ids['BDL'] = [s for s in df_demographic['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids['Census']['1988'] = [s for s in df_c_1988['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids['Census']['2002'] = [s for s in df_c_2002['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids['Census']['2011'] = [s for s in df_c_2011['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids['Census']['2021'] = [s for s in df_c_2021['subjectId'].unique() if s not in IGNORED_SUBJECTS]

subject_ids_flat = (subject_ids['BDL'] + subject_ids['Census']['1988'] +
                    subject_ids['Census']['2002'] + subject_ids['Census']['2011'] +
                    subject_ids['Census']['2021'])

subject_names_dict = {s: '' for s in subject_ids_flat}

# BDL
subject_names_dict['P2137'] = 'pop__age_sex'
# Census 1988
subject_names_dict['P2884'] = 'pop__age'
subject_names_dict['P2885'] = 'pop__educ'
subject_names_dict['P2883'] = 'pop__sex'
subject_names_dict['P2887'] = 'hh_size'
# Census 2002
subject_names_dict['P2114'] = 'pop__age_sex'
subject_names_dict['P2403'] = 'pop__age_educ'
subject_names_dict['P2402'] = 'pop__sex_educ'
subject_names_dict['P2871'] = 'hh_size'
# Census 2011
subject_names_dict['P3304'] = 'pop__age_sex'
subject_names_dict['P3311'] = 'pop__age_educ'
subject_names_dict['P3309'] = 'pop__sex_educ'
subject_names_dict['P3420'] = 'hh_size'
# Census 2021
subject_names_dict['P4253'] = 'pop__age_sex'
subject_names_dict['P4320'] = 'pop__age_educ'
subject_names_dict['P4315'] = 'pop__sex_educ'
subject_names_dict['P4287'] = 'hh_size'

print(f'Subject IDs collected: BDL={len(subject_ids["BDL"])}, Census total={sum(len(v) for v in subject_ids["Census"].values())}')
print(f'Subject names dict: {len(subject_names_dict)} entries')

In [10]:
# =============================================================================
# STEP 7: Process ALL Subjects
# =============================================================================
df_subjects = {
    'BDL': df_demographic,
    'Census': {'1988': df_c_1988, '2002': df_c_2002, '2011': df_c_2011, '2021': df_c_2021}
}

df_processed_subjects = {
    'BDL': {},
    'Census': {'1988': {}, '2002': {}, '2011': {}, '2021': {}}
}

# --- Process BDL and Census subjects ---
for subject in subject_ids.items():
    if subject[0] == 'BDL':
        for s in subject[1]:
            print(f'Processing BDL subject: {s}...')
            df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
            df_processed_subjects['BDL'][s] = df
    else:
        for sub in subject[1].items():
            year = sub[0]
            df_c = df_subjects['Census'][year]
            for s in sub[1]:
                print(f'  Processing Census {year} subject: {s}...')
                df = gtdb.GeoTERYTDatabase.process_subject_data(df_c, df_c_variables, s)
                df_processed_subjects['Census'][year][s] = df

# --- P4315: Convert wide-format Census 2021 pop__sex_educ ---
print('\nProcessing P4315 (wide-format Census 2021 pop__sex_educ)...')
rows = []
var_id_counter = 900001
var_id_lookup = {}
for col in df_c_2021_add.columns:
    if col in ['Kod', 'Nazwa']:
        continue
    parts = col.split(';')
    if len(parts) < 3:
        continue
    sex = parts[0].strip()
    educ = parts[1].strip()
    year = int(parts[2].strip())
    key = (sex, educ)
    if key not in var_id_lookup:
        var_id_lookup[key] = var_id_counter
        var_id_counter += 1
    vid = var_id_lookup[key]
    for _, row in df_c_2021_add.iterrows():
        kod = str(row['Kod']).zfill(7)
        val = row[col]
        if pd.isna(val):
            continue
        rows.append({
            'nuts_id': kod.ljust(12, '0'), 'name': row['Nazwa'],
            'variableId': vid, 'subjectId': 'P4315', 'var_id': vid,
            'n1': sex, 'n2': educ, 'year': year, 'val': float(val), 'teryt_id': kod
        })
df_p4315 = pd.DataFrame(rows)
df_processed_subjects['Census']['2021']['P4315'] = df_p4315
if 'P4315' not in subject_ids['Census']['2021']:
    subject_ids['Census']['2021'].append('P4315')
print(f'  P4315: {df_p4315.shape[0]:,} rows, {df_p4315["teryt_id"].nunique()} units')

# --- Fix hh_size subjects ---
# P3420 (Census 2011): filter + relabel
if 'P3420' in df_processed_subjects['Census']['2011']:
    df = df_processed_subjects['Census']['2011']['P3420']
    before = len(df)
    if 'n2' in df.columns:
        df = df[df['n2'] == 'wartość liczbowa'].copy()
        if df['n2'].nunique() <= 1:
            df = df.drop(columns=['n2'])
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
        label_map = {
            'osoby w gospodarstwie domowym - 1': '1-osobowe',
            'osoby w gospodarstwie domowym - 2': '2-osobowe',
            'osoby w gospodarstwie domowym - 3': '3-osobowe',
            'osoby w gospodarstwie domowym - 4': '4-osobowe',
            'osoby w gospodarstwie domowym - 5 i więcej': '5-osobowe i większe',
        }
        df['n1'] = df['n1'].replace(label_map)
    df_processed_subjects['Census']['2011']['P3420'] = df
    print(f'  P3420 fix: {before} -> {len(df)} rows')

# P2871 (Census 2002): keep only 'gospodarstwa'
if 'P2871' in df_processed_subjects['Census']['2002']:
    df = df_processed_subjects['Census']['2002']['P2871']
    before = len(df)
    if 'n1' in df.columns:
        df = df[df['n1'] == 'gospodarstwa'].copy()
        if df['n1'].nunique() <= 1:
            df = df.drop(columns=['n1'])
    df_processed_subjects['Census']['2002']['P2871'] = df
    print(f'  P2871 fix: {before} -> {len(df)} rows')

# P4287 (Census 2021): remove useless variables
if 'P4287' in df_processed_subjects['Census']['2021']:
    df = df_processed_subjects['Census']['2021']['P4287']
    before = len(df)
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
    df_processed_subjects['Census']['2021']['P4287'] = df
    print(f'  P4287 fix: {before} -> {len(df)} rows')

del df_subjects
gc.collect()
print('\n✓ All standard subjects processed')

  Processing Census 1988 subject: P2884...
  Processing Census 1988 subject: P2885...
  Processing Census 1988 subject: P2883...
  Processing Census 1988 subject: P2887...
  Processing Census 2002 subject: P2114...
  Processing Census 2002 subject: P2403...
  Processing Census 2002 subject: P2402...
  Processing Census 2002 subject: P2871...
  Processing Census 2011 subject: P3304...
  Processing Census 2011 subject: P3311...
  Processing Census 2011 subject: P3309...
  Processing Census 2011 subject: P3420...
  Processing Census 2021 subject: P4253...
  Processing Census 2021 subject: P4320...
  Processing Census 2021 subject: P4287...

Processing P4315 (wide-format Census 2021 pop__sex_educ)...
  P4315: 125,880 rows, 4196 units
  P3420 fix: 6336 -> 2376 rows
  P2871 fix: 48540 -> 24270 rows
  P4287 fix: 29379 -> 20985 rows

✓ All standard subjects processed


In [11]:
# =============================================================================
# STEP 8: Process P2350 & P4092 (wide-format BDL education data)
# =============================================================================

def bdl_kod_to_teryt(kod):
    """Convert 12-digit BDL Kod to 7-digit TERYT ID."""
    kod_str = str(int(kod)).zfill(11)
    if kod_str == '000000000000':
        return '0000000'
    sub = kod_str[1:5]
    if sub == '1410':
        return '1300000'
    elif sub == '1420':
        return '1500000'
    woj = kod_str[1:3]
    return woj + '00000'

def process_wide_educ_csv(df_wide, subject_id, set_nan_year=None):
    """Convert wide-format BDL education CSV to long format."""
    rows = []
    var_id_counter = 800001
    var_id_lookup = {}
    for col in df_wide.columns:
        if col in ['Kod', 'Nazwa'] or pd.isna(col) or str(col).strip() == '':
            continue
        parts = str(col).split(';')
        if len(parts) < 4:
            continue
        educ = parts[1].strip()
        year = int(parts[3].strip())
        if educ not in var_id_lookup:
            var_id_lookup[educ] = var_id_counter
            var_id_counter += 1
        vid = var_id_lookup[educ]
        for _, row in df_wide.iterrows():
            teryt_id = bdl_kod_to_teryt(row['Kod'])
            val = row[col]
            if pd.isna(val):
                continue
            val = float(val) * 1000
            if set_nan_year and year == set_nan_year:
                val = np.nan
            if not pd.isna(val) and val == 0:
                val = np.nan
            rows.append({
                'teryt_id': teryt_id, 'name': row['Nazwa'],
                'variableId': vid, 'subjectId': subject_id, 'var_id': vid,
                'n1': educ, 'year': year, 'val': val
            })
    df_long = pd.DataFrame(rows)
    interpolated_rows = []
    for (tid, educ), grp in df_long.groupby(['teryt_id', 'n1']):
        grp_sorted = grp.sort_values('year')
        series = grp_sorted.set_index('year')['val']
        series = series.interpolate(method='linear')
        for yr, v in series.items():
            if pd.isna(v):
                continue
            ref_row = grp_sorted[grp_sorted['year'] == yr].iloc[0]
            interpolated_rows.append({
                'teryt_id': tid, 'name': ref_row['name'],
                'variableId': ref_row['variableId'], 'subjectId': subject_id,
                'var_id': ref_row['var_id'], 'n1': educ, 'year': yr, 'val': v
            })
    return pd.DataFrame(interpolated_rows)

print('Processing P2350 (BDL education 1995-2020)...')
df_p2350 = process_wide_educ_csv(df_educ_add_1, 'P2350', set_nan_year=2000)
print(f'  P2350: {len(df_p2350):,} rows, {df_p2350["teryt_id"].nunique()} units')

print('Processing P4092 (BDL education 2010-2024)...')
df_p4092 = process_wide_educ_csv(df_educ_add_2, 'P4092')
print(f'  P4092: {len(df_p4092):,} rows, {df_p4092["teryt_id"].nunique()} units')

df_processed_subjects['BDL']['P2350'] = df_p2350
df_processed_subjects['BDL']['P4092'] = df_p4092
if 'P2350' not in subject_ids['BDL']:
    subject_ids['BDL'].append('P2350')
if 'P4092' not in subject_ids['BDL']:
    subject_ids['BDL'].append('P4092')
subject_names_dict['P2350'] = 'pop__educ'
subject_names_dict['P4092'] = 'pop__educ'

print(f'\n  P2350 labels: {sorted(df_p2350["n1"].unique())}')
print(f'  P4092 labels: {sorted(df_p4092["n1"].unique())}')

Processing P2350 (BDL education 1995-2020)...
  P2350: 2,410 rows, 19 units
Processing P4092 (BDL education 2010-2024)...
  P4092: 1,425 rows, 19 units

  P2350 labels: ['gimnazjalne, podstawowe i niższe', 'policealne oraz średnie zawodowe/branżowe', 'wyższe', 'zasadnicze zawodowe/branżowe', 'średnie ogólnokształcące']
  P4092 labels: ['gimnazjalne, podstawowe i niższe', 'policealne oraz średnie zawodowe/branżowe', 'wyższe', 'zasadnicze zawodowe/branżowe', 'średnie ogólnokształcące']


In [12]:
# =============================================================================
# STEP 9: Load Subject Data onto TERYTRecords
# =============================================================================
for subject in subject_ids.items():
    if subject[0] == 'BDL':
        for s in subject[1]:
            print(f'Loading BDL subject: {s}...')
            df = df_processed_subjects['BDL'][s]
            stats = db.load_subject_data(df, source_type='BDL', subject_id=s,
                                         subject_name=subject_names_dict[s])
            print(f'  {stats["matched_teryts"]} matched, {stats["total_data_points"]:,} points')
    else:
        for sub in subject[1].items():
            year = sub[0]
            for s in sub[1]:
                print(f'  Loading Census {year} subject: {s}...')
                df = df_processed_subjects['Census'][year][s]
                stats = db.load_subject_data(df, source_type='Census', subject_id=s,
                                             subject_name=subject_names_dict[s])
                print(f'    {stats["matched_teryts"]} matched, {stats["total_data_points"]:,} points')

del df_processed_subjects
gc.collect()

summary = db.get_data_summary()
print(f'\n✓ Data loaded: {summary["records_with_data"]} records, {summary["total_data_points"]:,} points')
print(f'  Subjects: {summary["subjects"]}')

Loading BDL subject: P2137...
  ✓ Loaded 7,183,824 data points for subject P2137
  ✓ Matched 4550 TERYT records, 6 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991', '1465998']
  4550 matched, 7,183,824 points
Loading BDL subject: P2350...
  ✓ Loaded 2,410 data points for subject P2350
  ✓ Matched 19 TERYT records, 0 unmatched
  19 matched, 2,410 points
Loading BDL subject: P4092...
  ✓ Loaded 1,425 data points for subject P4092
  ✓ Matched 19 TERYT records, 0 unmatched
  19 matched, 1,425 points
  Loading Census 1988 subject: P2884...
  ✓ Loaded 32,112 data points for subject P2884
  ✓ Matched 4014 TERYT records, 5 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991']
    4014 matched, 32,112 points
  Loading Census 1988 subject: P2885...
  ✓ Loaded 16,056 data points for subject P2885
  ✓ Matched 4014 TERYT records, 5 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', 

In [13]:
# =============================================================================
# STEP 10: Load Historical Cross Table Data (from GUS02A pickles)
# =============================================================================
with open(gus_root / 'pop_age_cross_tables.pkl', 'rb') as f:
    age_sex_cross_tables = pickle.load(f)
with open(gus_root / 'pop_educ_cross_tables.pkl', 'rb') as f:
    educ_sex_cross_tables = pickle.load(f)

print(f'Age-sex cross tables: {len(age_sex_cross_tables)} voivodeships')
print(f'Educ-sex cross tables: {len(educ_sex_cross_tables)} years')

# Build voivodeship name -> teryt_id mapping
name_to_teryt = {}
old_voiv_gdf = db.get_old_voivodship_gdf()
if old_voiv_gdf is not None:
    for _, row in old_voiv_gdf.iterrows():
        name_cap = str(row['name']).capitalize()
        name_to_teryt[name_cap] = row['teryt_id']
        name_to_teryt[str(row['name'])] = row['teryt_id']
name_to_teryt['Polska'] = '0000000'

# --- 1. Age x Sex Cross Tables (old voivodeships, 1986-1994) ---
print('\n=== Processing Age x Sex Historical Data ===')
rows_age_sex = []
vid_counter = 700001
vid_lookup = {}
unmatched = set()
for voiv_name, years_dict in age_sex_cross_tables.items():
    teryt_id = name_to_teryt.get(voiv_name)
    if not teryt_id:
        for k, v in name_to_teryt.items():
            if k.lower() == voiv_name.lower():
                teryt_id = v
                break
    if not teryt_id:
        unmatched.add(voiv_name)
        continue
    for year_str, df in years_dict.items():
        year = int(year_str)
        for age_group in df.index:
            for sex in df.columns:
                key = (age_group, sex)
                if key not in vid_lookup:
                    vid_lookup[key] = vid_counter
                    vid_counter += 1
                vid = vid_lookup[key]
                val = df.loc[age_group, sex]
                if pd.isna(val):
                    continue
                rows_age_sex.append({
                    'teryt_id': teryt_id, 'name': voiv_name,
                    'variableId': vid, 'subjectId': 'H_age_sex',
                    'var_id': vid, 'n1': age_group, 'n2': sex,
                    'year': year, 'val': float(val)
                })

df_h_age_sex = pd.DataFrame(rows_age_sex)
print(f'  H_age_sex: {len(df_h_age_sex):,} rows, {df_h_age_sex["teryt_id"].nunique()} units')
if unmatched:
    print(f'  ⚠ Unmatched: {unmatched}')

# --- 2. Sex x Education Cross Tables (country level) ---
print('\n=== Processing Sex x Education Historical Data ===')
rows_sex_educ = []
rows_educ_age = []
vid_counter_se = 750001
vid_lookup_se = {}
vid_counter_ea = 760001
vid_lookup_ea = {}
sex_key_map = {'ogołem': 'ogółem', 'meżczyzni': 'mężczyźni', 'kobiety': 'kobiety'}

for year_str, year_data in educ_sex_cross_tables.items():
    year = int(year_str)
    if 'sex_education' in year_data:
        df = year_data['sex_education']
        for sex in df.index:
            for educ in df.columns:
                key = (sex, educ)
                if key not in vid_lookup_se:
                    vid_lookup_se[key] = vid_counter_se
                    vid_counter_se += 1
                vid = vid_lookup_se[key]
                val = df.loc[sex, educ]
                if pd.isna(val):
                    continue
                rows_sex_educ.append({
                    'teryt_id': '0000000', 'name': 'Polska',
                    'variableId': vid, 'subjectId': 'H_sex_educ',
                    'var_id': vid, 'n1': sex, 'n2': educ,
                    'year': year, 'val': float(val)
                })
    if 'education_age' in year_data:
        for raw_sex_key, df in year_data['education_age'].items():
            sex = sex_key_map.get(raw_sex_key, raw_sex_key)
            for educ in df.index:
                for age in df.columns:
                    key = (sex, educ, age)
                    if key not in vid_lookup_ea:
                        vid_lookup_ea[key] = vid_counter_ea
                        vid_counter_ea += 1
                    vid = vid_lookup_ea[key]
                    val = df.loc[educ, age]
                    if pd.isna(val):
                        continue
                    rows_educ_age.append({
                        'teryt_id': '0000000', 'name': 'Polska',
                        'variableId': vid, 'subjectId': 'H_educ_age',
                        'var_id': vid, 'n1': sex, 'n2': educ, 'n3': age,
                        'year': year, 'val': float(val)
                    })

df_h_sex_educ = pd.DataFrame(rows_sex_educ)
df_h_educ_age = pd.DataFrame(rows_educ_age) if rows_educ_age else pd.DataFrame()
print(f'  H_sex_educ: {len(df_h_sex_educ):,} rows')
if len(df_h_educ_age) > 0:
    print(f'  H_educ_age: {len(df_h_educ_age):,} rows')

# --- 3. Load onto database ---
print('\n=== Loading Historical Data onto Database ===')
stats = db.load_subject_data(df_h_age_sex, source_type='Historical',
                             subject_id='H_age_sex', subject_name='pop__age_sex')
print(f'  H_age_sex: {stats["matched_teryts"]} matched, {stats["total_data_points"]:,} points')

stats = db.load_subject_data(df_h_sex_educ, source_type='Historical',
                             subject_id='H_sex_educ', subject_name='pop__sex_educ')
print(f'  H_sex_educ: {stats["matched_teryts"]} matched, {stats["total_data_points"]:,} points')

if len(df_h_educ_age) > 0:
    stats = db.load_subject_data(df_h_educ_age, source_type='Historical',
                                 subject_id='H_educ_age', subject_name='pop__educ_age')
    print(f'  H_educ_age: {stats["matched_teryts"]} matched, {stats["total_data_points"]:,} points')

del age_sex_cross_tables, educ_sex_cross_tables
gc.collect()

summary = db.get_data_summary()
print(f'\n✓ After historical loading: {summary["records_with_data"]} records, {summary["total_data_points"]:,} points')

Age-sex cross tables: 50 voivodeships
Educ-sex cross tables: 7 years

=== Processing Age x Sex Historical Data ===
  H_age_sex: 22,950 rows, 50 units

=== Processing Sex x Education Historical Data ===
  H_sex_educ: 108 rows
  H_educ_age: 135 rows

=== Loading Historical Data onto Database ===
  ✓ Loaded 22,950 data points for subject H_age_sex
  ✓ Matched 50 TERYT records, 0 unmatched
  H_age_sex: 50 matched, 22,950 points
  ✓ Loaded 108 data points for subject H_sex_educ
  ✓ Matched 1 TERYT records, 0 unmatched
  H_sex_educ: 1 matched, 108 points
  ✓ Loaded 135 data points for subject H_educ_age
  ✓ Matched 1 TERYT records, 0 unmatched
  H_educ_age: 1 matched, 135 points

✓ After historical loading: 4585 records, 8,419,003 points


In [14]:
# =============================================================================
# STEP 11: Resolve Historical TERYTs
# =============================================================================
importlib.reload(gtdb)

print('Resolving historical TERYT codes...')
recovery = db.resolve_historical_teryts(verbose=True)

total_recovered = sum(recovery.values())
print(f'\n✓ Total data points recovered: {total_recovered:,}')
for sid, n in sorted(recovery.items(), key=lambda x: -x[1])[:10]:
    print(f'  {sid}: {n:,} points')

Resolving historical TERYT codes...
Resolving historical TERYTs for 19 subjects...
  P2137: recovered 349848 data points
  ✓ Total recovered: 349848 data points across 1 subjects

✓ Total data points recovered: 349,848
  P2137: 349,848 points
  P2114: 0 points
  P2350: 0 points
  P2402: 0 points
  P2403: 0 points
  P2871: 0 points
  P2883: 0 points
  P2884: 0 points
  P2885: 0 points
  P2887: 0 points


In [15]:
# =============================================================================
# STEP 12: Extend Year Range to 1986
# =============================================================================
print('Extending year range...')
stats = db.extend_year_range(verbose=True)
print(f'\n✓ Extended {stats} records')

gc.collect()

Extending year range...
Extending year range from 1986-2025 to 1986-2025...
  ✓ Extended 4613 records to 1986-2025

✓ Extended None records


30

In [16]:
# =============================================================================
# STEP 13: Create ALL Merged Subjects
# =============================================================================
# Phase 1: Auto-merge subjects sharing the same name
# Phase 2: Custom M_ subjects (M_hh_size_1990, M_hh_size_2000, M_age_sex,
#          M_age_1990, M_educ_1990, M_educ_2000, M_educ_sex_1990, M_educ_sex_2000)

importlib.reload(gtdb)

print('Creating merged subjects...')
merged_info = db.create_merged_subjects(subject_names_dict)

# Update subject_names_dict with merged subjects
for merged_sid, source_ids in merged_info.items():
    group_name = merged_sid.replace('M_', '')
    subject_names_dict[merged_sid] = group_name

print(f'\n✓ Created {len(merged_info)} merged subjects:')
for sid, sources in merged_info.items():
    print(f'  {sid}: {sources}')

summary = db.get_data_summary()
print(f'\nAfter merge: {summary["records_with_data"]} records, {summary["total_data_series"]:,} series, {summary["total_data_points"]:,} points')

Creating merged subjects...
Phase 1 — Auto-merge: 5 subject groups
  pop__age_sex: ['P2137', 'P2114', 'P3304', 'P4253']
  pop__educ: ['P2885', 'P2350', 'P4092']
  hh_size: ['P2887', 'P2871', 'P3420', 'P4287']
  pop__age_educ: ['P2403', 'P3311', 'P4320']
  pop__sex_educ: ['P2402', 'P3309', 'P4315']

  Merged subject: M_pop__age_sex
    n1: 19 labels
    n2: 3 labels
    Aggregates in P2137/n1: {'70 i więcej', '0-14'}
    ✓ 4534 records, 257736 merged series

  Merged subject: M_pop__educ
    n1: 8 labels
    ✓ 4016 records, 16134 merged series

  Merged subject: M_hh_size
    n1: 5 labels
    ✓ 4014 records, 16056 merged series

  Merged subject: M_pop__age_educ
    n1: 11 labels
    n2: 17 labels
    ✓ 398 records, 74239 merged series

  Merged subject: M_pop__sex_educ
    n1: 3 labels
    n2: 16 labels
    ✓ 4277 records, 199176 merged series

Phase 2 — Custom merged subjects
  M_hh_size_1990: 36360 entries stored
  M_hh_size_2000: 46956 entries stored
  M_age_sex: 220032 entries stor

In [17]:
# =============================================================================
# STEP 14: Build Cross Tables for ALL Subjects
# =============================================================================
print('Building cross tables for all subjects...')
for s, name in subject_names_dict.items():
    n_built = db.build_cross_tables(s, subject_name=name)

# Also build for historical subjects not in subject_names_dict
for hsid in ['H_age_sex', 'H_sex_educ', 'H_educ_age']:
    if hsid not in subject_names_dict:
        n_built = db.build_cross_tables(hsid, subject_name=hsid)

ct_summary = db.get_cross_table_summary()
print(f'\nCross table summary:')
display(ct_summary)

Building cross tables for all subjects...
  ✓ Built 4534 cross tables for subject P2137 (pop__age_sex)
  ✓ Built 4014 cross tables for subject P2884 (pop__age)
  ✓ Built 4014 cross tables for subject P2885 (pop__educ)
  ✓ Built 4014 cross tables for subject P2883 (pop__sex)
  ✓ Built 4014 cross tables for subject P2887 (hh_size)
  ✓ Built 4045 cross tables for subject P2114 (pop__age_sex)
  ✓ Built 397 cross tables for subject P2403 (pop__age_educ)
  ✓ Built 4045 cross tables for subject P2402 (pop__sex_educ)
  ✓ Built 4045 cross tables for subject P2871 (hh_size)
  ✓ Built 4097 cross tables for subject P3304 (pop__age_sex)
  ✓ Built 396 cross tables for subject P3311 (pop__age_educ)
  ✓ Built 396 cross tables for subject P3309 (pop__sex_educ)
  ✓ Built 396 cross tables for subject P3420 (hh_size)
  ✓ Built 4196 cross tables for subject P4253 (pop__age_sex)
  ✓ Built 397 cross tables for subject P4320 (pop__age_educ)
  ✓ Built 4196 cross tables for subject P4287 (hh_size)
  ✓ Built 419

,subject_name,n_records,dimensions,shape
subject_id,,,,
P2137,pop__age_sex,4534,n1 × n2,"(21, 3)"
P2884,pop__age,4014,n1,"(8,)"
P2885,pop__educ,4014,n1,"(4,)"
P2883,pop__sex,4014,n1,"(3,)"
P2887,hh_size,4014,n1,"(4,)"
P2114,pop__age_sex,4045,n1 × n2,"(3, 19)"
P2403,pop__age_educ,397,n1 × n2,"(12, 8)"
P2402,pop__sex_educ,4045,n1 × n2,"(3, 8)"
P2871,hh_size,4045,n2,"(6,)"


In [18]:
# =============================================================================
# STEP 15: Extract Population, Classify Urban/Rural, Code Labels
# =============================================================================
print('Extracting total population...')
n_pop = db.extract_population(subject_names_dict)

print('\nClassifying urban/rural...')
n_class = db.classify_population()

print('\nCoding dimension labels...')
n_coded = db.code_dimension_labels(subject_names_dict)

Extracting total population...
  ✓ Phase 1: population from DataSeries for 4534 records
  ✓ Phase 2: population from cross tables for 50 additional records
  ✓ Total records with population data: 4583

Classifying urban/rural...
  ✓ Classified 3412 records by urban/rural

Coding dimension labels...
  ✓ Coded dimension labels for 2533166 DataSeries across 35 subjects


In [19]:
# =============================================================================
# STEP 16: Save Complete Database
# =============================================================================
save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)
print(f'✓ Saved to {save_path}')

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  ✓ Saved 4613 records
  ✓ Records with data: 4585
  ✓ Records with cross tables: 4585
  ✓ File size: 2726.4 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl
✓ Saved to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl


In [ ]:
# =============================================================================
# STEP 17: Verify Data
# =============================================================================
data_summary = db.get_data_summary()
print('Data Summary:')
for k, v in data_summary.items():
    print(f'  {k}: {v}')

print('\nSubjects:')
for sid in sorted(data_summary['subjects']):
    sname = subject_names_dict.get(sid, '')
    prefix = 'MERGED' if sid.startswith('M_') else 'RAW'
    print(f'  [{prefix}] {sid}: {sname}')

# Quick spot check
import random
random.seed(42)
candidates = [r for r in db._records.values() if r.level == 6 and r.has_data and r.n_data_series > 50]
sample_rec = random.choice(candidates)
print(f'\nSample record: {sample_rec.teryt_id} - {sample_rec.name}')
print(f'  Data series: {sample_rec.n_data_series}')
print(f'  Subjects: {sample_rec.list_subjects()}')
print(f'  Cross tables: {sample_rec.list_cross_tables()}')

# Check M_ subjects exist
expected_m = ['M_age_sex', 'M_age_1990', 'M_hh_size_1990', 'M_hh_size_2000',
              'M_educ_1990', 'M_educ_2000', 'M_educ_sex_1990', 'M_educ_sex_2000']
for sid in expected_m:
    count = sum(1 for r in db._records.values() if r.get_data_by_subject(sid))
    print(f'  {sid}: {count} records with data')